In [ ]:
import numpy as np
from datasets import load_dataset, get_dataset_config_names
import random

# =======================================================================
# 🗺️ 데이터셋 정보 분석: Leorasz/military_bases
# 제목: 군사 기지 위치 및 정보 데이터셋
# 의미: 전 세계 여러 지역의 군사 기지(Military Bases)에 대한 지리적, 운영 정보를 담고 있습니다.
# 설명: 이 데이터는 기지 이름, 위치 국가, 운영 상태, 그리고 지리적 면적(ShapeSTArea, ShapeSTLength)과 같은 구조화된 메타데이터를 포함하고 있습니다.
# 💡 초보자 실습 목표: 데이터셋에서 특정 기준(예: 최대 면적)에 따라 가장 중요한 기지들을 '선별'하고 보고서 형태로 요약해 봅시다!
# =======================================================================

# 상수를 설정합니다.
DATASET_NAME = "Leorasz/military_bases"
SPLIT_NAME = "train"
SAMPLE_COUNT = 50  # 데이터셋 전체를 로드하지 않고, 재미있는 실습을 위해 50개의 샘플만 사용하겠습니다!

print("✨ 안녕하세요! AI 코딩 탐험가님, 준비되셨나요? 함께 데이터의 보물을 찾아 떠나볼까요! ✨")
print("-" * 70)

# 1. 데이터셋 Config 확인 (필수 요구사항 21)
try:
    configs = get_dataset_config_names(DATASET_NAME)
    print(f"✅ 사용 가능한 Config 목록: {configs}")
    # 해당 데이터셋은 Config가 없거나 기본 설정만 사용하므로, 첫 번째 Config를 선택합니다.
    selected_config = configs[0] if configs else None
except Exception as e:
    print(f"ℹ️ Config 확인 중 오류가 발생했거나 기본 설정만 제공됩니다: {e}")
    selected_config = None

# 2. 데이터셋 로드 (스트리밍 vs. 일반 모드)
print("\n🚀 데이터셋을 효율적으로 로드합니다...")
dataset = None
sample_iterator = None

# 스트리밍 모드로 먼저 시도합니다. (데이터가 매우 클 경우 메모리 효율적!)
try:
    # load_dataset(DATASET_NAME, split='...', streaming=True) 방식 사용
    dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME, streaming=True)
    print("✅ 성공! 스트리밍 모드 (Streaming=True)로 데이터셋을 로드했습니다. 메모리 걱정 끝!")
except Exception as e:
    # 스트리밍 모드가 실패하면 (일부 환경에서 발생 가능) 일반 모드로 전환합니다.
    print(f"⚠️ 스트리밍 로드에 실패했습니다 ({type(e).__name__}). 일반 모드로 전환하여 소량 다운로드합니다.")
    try:
        # streaming=False로 설정하고, 적은 양의 데이터만 다운로드합니다.
        dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME, streaming=False)
    except Exception as e_fail:
        print(f"🛑 데이터셋 로드에 심각한 오류가 발생했습니다. {e_fail}")
        exit()


# 3. 샘플링 로직 구현 (가장 중요한 부분!)
print("\n🌟 데이터 분석을 위해 상위 샘플들만 뽑아옵니다. (K=50)")

# 필수 요구사항 9번 패턴 사용
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)
    # next()를 사용할 준비를 합니다.
    dataset_iterator = dataset.take(SAMPLE_COUNT)
    
    # 데이터를 실제로 소비 가능한 리스트로 만듭니다. (필수 요구사항 16번 패턴)
    sample_data_list = list(dataset_iterator)
    
else:
    # 일반 데이터셋 (Dataset)
    # 일반 데이터셋인 경우, 직접 슬라이싱하여 리스트로 만듭니다.
    # 이 데이터셋은 list()로 변환해도 문제가 없다고 가정합니다.
    sample_data_list = list(dataset.select(range(min(SAMPLE_COUNT, len(dataset)))))

if not sample_data_list:
    print("🚨 로드된 샘플 데이터가 없습니다. 스크립트를 종료합니다.")
    exit()

# 4. 데이터 분석 실습: '최대 규모의 군사 기지' 찾기! (창의적 분석)
print(f"\n--- 🔎 분석 시작: 상위 {min(SAMPLE_COUNT, len(sample_data_list))}개 샘플 기반 분석 ---")

# 분석에 사용할 데이터를 저장할 리스트
site_metrics = []

# 샘플 데이터들을 순회하며 필요한 정보를 추출합니다.
for sample in sample_data_list:
    try:
        # 안전한 값 추출 (키가 없을 경우 대비)
        country = sample.get("countryName", "Unknown")
        site_name = sample.get("siteName", "Unnamed Site")
        area = sample.get("ShapeSTArea", 0.0)
        status = sample.get("siteOperationalStatus", "Status Unknown")
        
        # 분석에 사용할 메트릭을 저장합니다.
        site_metrics.append({
            "site_name": site_name,
            "country": country,
            "area": area,
            "status": status
        })
    except Exception as e:
        print(f"🚨 데이터 처리 중 오류 발생: {e}")
        continue

# 데이터 분석: 면적(Area)을 기준으로 내림차순 정렬하여 상위 기지들을 찾습니다.
# Python의 sorted() 함수와 람다식을 사용하여 원하는 키(area)를 기준으로 정렬합니다.
sorted_sites = sorted(site_metrics, key=lambda x: x['area'], reverse=True)

# 상위 5개 기지를 추출합니다.
top_k = 5
top_bases = sorted_sites[:top_k]


# 5. 결과 시각화 및 보고서 출력 (친절한 튜터의 보고서 작성)
print("\n" + "✨" * 70)
print("🏆 분석 결과: 가장 크고 규모가 큰 상위 기지 TOP 5 보고서 🏆")
print("✨" * 70)

if top_bases:
    # 표 형식의 출력을 위해 헤더를 먼저 출력합니다.
    print(f"{'순위':<5} | {'기지 이름 (Site Name)':<30} | {'국가 (Country)':<20} | {'면적 (Area - Sq Units)':<25} | {'운영 상태':<15}")
    print("-" * 110)
    
    # 상위 N개 기지를 반복하며 분석 결과를 출력합니다.
    for i, base in enumerate(top_bases):
        rank = i + 1
        print(f"{rank:<5} | {base['site_name']:<30} | {base['country']:<20} | {base['area']:.2f}{'':<15} | {base['status']:<15}")
else:
    print("😢 분석할 수 있는 유의미한 데이터를 찾지 못했어요. 데이터셋을 다시 확인해 주세요!")

print("\n----------------------------------------------------------------------")
print("🎉 축하합니다! 당신은 데이터의 메타정보를 성공적으로 분석하고 구조화된 보고서를 작성했습니다.")
print("이 과정이 AI 데이터 분석의 가장 기본적인 '탐색적 분석(EDA)' 단계입니다! 정말 대단해요!")
print("----------------------------------------------------------------------")